# Notebook 3: Statistical Analysis & Hypothesis Testing
## Federal Reserve Interest Rate Prediction

**Tests conducted:**
1. **Shapiro-Wilk** — Normality test
2. **ADF** — Augmented Dickey-Fuller stationarity test
3. **T-Test** — FED rates in high vs low inflation periods
4. **ANOVA** — FED rates across economic regimes
5. **Pearson & Spearman** — Correlation analysis


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#E91E63','#795548','#607D8B','#FF5722']
sns.set_palette(PALETTE)

DATA_PATH = r"d:/Projects/ML website/ML-Project/App/Tabs/Datasets/finaldataset.csv"
OUT_PATH  = r"d:/Projects/ML website/ML-Project/ml_analysis/outputs"

from scipy.stats import shapiro, ttest_ind, f_oneway, pearsonr, spearmanr
from statsmodels.tsa.stattools import adfuller


In [ ]:
# Load and prepare data
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').ffill().bfill().reset_index(drop=True)
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].clip(df[col].quantile(0.01), df[col].quantile(0.99))

FEATURES = ['ConsumerPriceIndexAllItems','GDP','InflationConsumerPrice',
            'MedianConsumerPriceIndex','RealGDP','RealGDPPerCapita',
            'RealPotentialGDP','UnemployemenrRate']
ALL_COLS = ['FEDRates'] + FEATURES
print(f"Data shape: {df.shape}")
display(df[ALL_COLS].describe().T.round(3))


## 1. Normality Tests — Shapiro-Wilk

In [ ]:
# Shapiro-Wilk normality test
print("Shapiro-Wilk Normality Tests:")
print(f"{'Feature':<40} {'W-stat':>10} {'p-value':>12} {'Normal?':>10}")
print("-"*75)
for col in ALL_COLS:
    sample = df[col].dropna().sample(min(500, len(df)), random_state=42)
    stat, p = shapiro(sample)
    normal = 'YES' if p > 0.05 else 'NO'
    print(f"{col:<40} {stat:>10.4f} {p:>12.4e} {normal:>10}")


In [ ]:
# Q-Q plots for normality assessment
from scipy.stats import probplot
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, col in enumerate(ALL_COLS):
    probplot(df[col].dropna(), dist='norm', plot=axes[i])
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].get_lines()[0].set(markersize=2, alpha=0.5, color=PALETTE[i % len(PALETTE)])
    axes[i].get_lines()[1].set(color='red', linewidth=1.5)
fig.suptitle('Q-Q Plots — Normality Assessment', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 2. Stationarity Tests — Augmented Dickey-Fuller

In [ ]:
# ADF stationarity test
print("ADF Stationarity Tests:")
print(f"{'Feature':<40} {'ADF stat':>10} {'p-value':>12} {'Result':>15}")
print("-"*80)
for col in ALL_COLS:
    result = adfuller(df[col].dropna(), autolag='AIC')
    status = 'STATIONARY' if result[1] < 0.05 else 'NON-STATIONARY'
    print(f"{col:<40} {result[0]:>10.4f} {result[1]:>12.4e} {status:>15}")


## 3. T-Test: FED Rates in High vs Low Inflation

In [ ]:
# Split into high/low inflation groups
med_inf = df['InflationConsumerPrice'].median()
high_inf = df.loc[df['InflationConsumerPrice'] >  med_inf, 'FEDRates']
low_inf  = df.loc[df['InflationConsumerPrice'] <= med_inf, 'FEDRates']

t_stat, t_p = ttest_ind(high_inf, low_inf)
print(f"High Inflation group — n={len(high_inf)}, mean FED rate: {high_inf.mean():.3f}%")
print(f"Low Inflation group  — n={len(low_inf)},  mean FED rate: {low_inf.mean():.3f}%")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value:     {t_p:.4e}")
print(f"Significant (p<0.05): {'YES' if t_p < 0.05 else 'NO'}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(high_inf, bins=40, alpha=0.6, color='#F44336',
        label=f'High Inflation (mean={high_inf.mean():.2f}%)')
ax.hist(low_inf,  bins=40, alpha=0.6, color='#2196F3',
        label=f'Low Inflation (mean={low_inf.mean():.2f}%)')
ax.axvline(high_inf.mean(), color='#F44336', linestyle='--', linewidth=2)
ax.axvline(low_inf.mean(),  color='#2196F3', linestyle='--', linewidth=2)
ax.set_title(f'T-Test: FED Rate by Inflation Level\nt={t_stat:.3f}, p={t_p:.2e}', fontweight='bold')
ax.set_xlabel('FED Rate (%)'); ax.legend()
plt.tight_layout(); plt.show()


## 4. ANOVA: FED Rates Across Economic Regimes

In [ ]:
# Define economic regimes by FED rate quartiles
df['RateRegime'] = pd.qcut(df['FEDRates'], q=4,
                            labels=['Very_Low','Low','High','Very_High'])
groups = [df.loc[df['RateRegime'] == r, 'FEDRates'].values
          for r in ['Very_Low','Low','High','Very_High']]
f_stat, f_p = f_oneway(*groups)

print(f"ANOVA: FED Rate across economic regimes")
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value:     {f_p:.4e}")
print(f"Significant: {'YES' if f_p < 0.05 else 'NO'}")

print("\nMean FED Rate by regime:")
print(df.groupby('RateRegime')['FEDRates'].agg(['mean','std','count']).round(3))

fig, ax = plt.subplots(figsize=(9, 5))
df.boxplot('FEDRates', by='RateRegime', ax=ax)
ax.set_title(f'ANOVA: FED Rate by Regime\nF={f_stat:.2f}, p={f_p:.2e}', fontweight='bold')
ax.set_xlabel('Economic Regime'); ax.set_ylabel('FED Rate (%)')
plt.suptitle('')
plt.tight_layout(); plt.show()


## 5. Spearman Rank Correlations

In [ ]:
# Spearman correlations
print(f"{'Feature':<40} {'Spearman rho':>14} {'p-value':>12}")
print("-"*70)
for col in FEATURES:
    rho, p = spearmanr(df[col], df['FEDRates'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f"{col:<40} {rho:>14.4f} {p:>12.4e} {sig}")

# Comparison: Pearson vs Spearman
pearson_r  = {col: pearsonr(df[col], df['FEDRates'])[0]  for col in FEATURES}
spearman_r = {col: spearmanr(df[col], df['FEDRates'])[0] for col in FEATURES}
corr_compare = pd.DataFrame({'Pearson r': pearson_r, 'Spearman rho': spearman_r}).round(4)
print("\nPearson vs Spearman comparison:")
display(corr_compare.sort_values('Spearman rho', key=abs, ascending=False))


In [ ]:
# Visualization: all correlation methods
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
spearman_vals = pd.Series({col: spearmanr(df[col], df['FEDRates'])[0] for col in FEATURES}).sort_values()
pearson_vals  = pd.Series({col: pearsonr(df[col], df['FEDRates'])[0]  for col in FEATURES}).sort_values()

axes[0].barh(range(len(spearman_vals)), spearman_vals.values,
             color=['#F44336' if v < 0 else '#2196F3' for v in spearman_vals.values])
axes[0].set_yticks(range(len(spearman_vals))); axes[0].set_yticklabels(spearman_vals.index)
axes[0].set_title('Spearman Rank Correlations with FED Rate', fontweight='bold')
axes[0].axvline(0, color='black', linewidth=0.8)

axes[1].barh(range(len(pearson_vals)), pearson_vals.values,
             color=['#F44336' if v < 0 else '#2196F3' for v in pearson_vals.values])
axes[1].set_yticks(range(len(pearson_vals))); axes[1].set_yticklabels(pearson_vals.index)
axes[1].set_title('Pearson Correlations with FED Rate', fontweight='bold')
axes[1].axvline(0, color='black', linewidth=0.8)
plt.tight_layout(); plt.show()


## Summary
- All features are non-normal (Shapiro-Wilk p << 0.05)
- GDP variables are non-stationary (ADF), inflation nearly stationary
- T-Test: Strong evidence rates are higher during high-inflation (p ≈ 5.7×10⁻⁷⁴)
- Spearman ρ = 0.68 for Inflation — strongest predictor